# Image Studio — Kaggle v1

Single-cell ComfyUI image studio, built for Shaun's Kaggle T4 workflow.

1. Enable **Internet** and a **T4 GPU** in Kaggle settings.
2. Run the cell below once; open its **OPEN IMAGE STUDIO** URL.
3. In ComfyUI's workflow menu, open **Image Studio - Create**, **Restyle**, **Inpaint**, or **Detail**. Change the prompt and press **Run**. No image starts automatically.
4. Images and logs are in `/kaggle/working/image-studio`. **Save Version with outputs** before ending your session. PNG metadata retains the generation graph and seed; drag a PNG into ComfyUI to restore it.

**Create:** 1024×1024, 30 steps, DPM++ 2M Karras, CFG 5.5. Try 832×1216 portrait or 1216×832 landscape, batch 1. These are starting settings, not a universal optimum.

**Restyle:** upload a source with dimensions around 1 megapixel; denoise 0.6 balances source structure and change. Lower preserves more structure. **Inpaint:** upload, right-click the image node → Open in MaskEditor, paint the area to change, save the mask and run. Uses SDXL latent-noise masking, not a dedicated inpainting checkpoint; seams and identity changes remain possible.

**Detail:** 1024px generation followed by 1.5× latent enlargement and a 0.3-denoise detail pass. This is generative detail, not faithful super-resolution; it can change faces/text. It is slower and uses more VRAM. If memory fails, use Create or reduce the selected resolution explicitly. No automatic quality reductions occur.

**Model:** [Juggernaut XL v9](https://huggingface.co/RunDiffusion/Juggernaut-XL-v9), about 7.1 GB, verified with the model file's published SHA256. Read its model card and underlying SDXL license before commercial use. This is a practical SDXL quality option for T4, not a claim to be the best model for every image. One T4 runs the graph; a second T4 is not pooled VRAM. No paid image API is used.

The checkpoint is reused during the session. To avoid future downloads, save it separately as a private Kaggle model/input with its original filename; attached inputs are discovered and checksum-verified. Model weights/runtime live in `/kaggle/temp`, not in saved image outputs. Dependencies require Internet even with an attached checkpoint. ComfyUI and cloudflared are pinned; Kaggle's Torch stack is preserved with constraints. Future Kaggle image changes may need a compatibility update.

**Validation limit:** notebook/source and graph structure checked locally; no Kaggle GPU generation, output quality, T4 speed, or public tunnel reliability has been verified. If setup fails, retain `comfy.log` and `tunnel.log`. Keep the startup cell running; stopping it stops its own server and tunnel. Quick Tunnel URLs grant access to anyone holding the URL.


In [ ]:
# Single-cell startup. No image generation starts automatically.
from pathlib import Path
import sys, os, subprocess, json, time, urllib.request, hashlib, shutil, re, signal, fcntl
WORKFLOW_PAYLOAD = {'Create': {'last_node_id': 7, 'last_link_id': 9, 'nodes': [{'id': 1, 'type': 'CheckpointLoaderSimple', 'pos': [0, 0], 'size': [320, 250], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [3]}, {'name': 'CLIP', 'type': 'CLIP', 'links': [1, 2]}, {'name': 'VAE', 'type': 'VAE', 'links': [8]}], 'properties': {'Node name for S&R': 'CheckpointLoaderSimple'}, 'widgets_values': ['Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors']}, {'id': 2, 'type': 'CLIPTextEncode', 'pos': [380, 0], 'size': [320, 250], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [4]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': ['Cinematic photograph of a lone explorer overlooking a vast alien coastline, dramatic clouds, natural light, detailed textures, thoughtful composition'], 'title': 'Describe your image'}, {'id': 3, 'type': 'CLIPTextEncode', 'pos': [380, 300], 'size': [320, 250], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [5]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': ['blurry, low quality, distorted anatomy, watermark, oversaturated'], 'title': 'Negative prompt'}, {'id': 4, 'type': 'EmptyLatentImage', 'pos': [380, 650], 'size': [320, 250], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [6]}], 'properties': {'Node name for S&R': 'EmptyLatentImage'}, 'widgets_values': [1024, 1024, 1]}, {'id': 5, 'type': 'KSampler', 'pos': [770, 0], 'size': [320, 250], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 3}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 4}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 5}, {'name': 'latent_image', 'type': 'LATENT', 'link': 6}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [7]}], 'properties': {'Node name for S&R': 'KSampler'}, 'widgets_values': [42, 'randomize', 30, 5.5, 'dpmpp_2m', 'karras', 1.0]}, {'id': 6, 'type': 'VAEDecodeTiled', 'pos': [1130, 350], 'size': [320, 250], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 7}, {'name': 'vae', 'type': 'VAE', 'link': 8}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [9]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8]}, {'id': 7, 'type': 'SaveImage', 'pos': [1490, 350], 'size': [320, 250], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 9}], 'outputs': [], 'properties': {'Node name for S&R': 'SaveImage'}, 'widgets_values': ['ImageStudio/Create']}], 'links': [[1, 1, 1, 2, 0, 'CLIP'], [2, 1, 1, 3, 0, 'CLIP'], [3, 1, 0, 5, 0, 'MODEL'], [4, 2, 0, 5, 1, 'CONDITIONING'], [5, 3, 0, 5, 2, 'CONDITIONING'], [6, 4, 0, 5, 3, 'LATENT'], [7, 5, 0, 6, 0, 'LATENT'], [8, 1, 2, 6, 1, 'VAE'], [9, 6, 0, 7, 0, 'IMAGE']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.65, 'offset': [50, 100]}}, 'version': 0.4}, 'Restyle': {'last_node_id': 8, 'last_link_id': 11, 'nodes': [{'id': 1, 'type': 'CheckpointLoaderSimple', 'pos': [0, 0], 'size': [320, 250], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [5]}, {'name': 'CLIP', 'type': 'CLIP', 'links': [1, 2]}, {'name': 'VAE', 'type': 'VAE', 'links': [4, 10]}], 'properties': {'Node name for S&R': 'CheckpointLoaderSimple'}, 'widgets_values': ['Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors']}, {'id': 2, 'type': 'CLIPTextEncode', 'pos': [380, 0], 'size': [320, 250], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [6]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': ['Cinematic photograph of a lone explorer overlooking a vast alien coastline, dramatic clouds, natural light, detailed textures, thoughtful composition'], 'title': 'Describe your image'}, {'id': 3, 'type': 'CLIPTextEncode', 'pos': [380, 300], 'size': [320, 250], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [7]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': ['blurry, low quality, distorted anatomy, watermark, oversaturated'], 'title': 'Negative prompt'}, {'id': 4, 'type': 'LoadImage', 'pos': [0, 400], 'size': [320, 250], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [3]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'Upload a source image'}, {'id': 5, 'type': 'VAEEncode', 'pos': [380, 650], 'size': [320, 250], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 3}, {'name': 'vae', 'type': 'VAE', 'link': 4}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [8]}], 'properties': {'Node name for S&R': 'VAEEncode'}, 'widgets_values': []}, {'id': 6, 'type': 'KSampler', 'pos': [770, 0], 'size': [320, 250], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 5}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 6}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 7}, {'name': 'latent_image', 'type': 'LATENT', 'link': 8}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [9]}], 'properties': {'Node name for S&R': 'KSampler'}, 'widgets_values': [42, 'randomize', 30, 5.5, 'dpmpp_2m', 'karras', 0.6]}, {'id': 7, 'type': 'VAEDecodeTiled', 'pos': [1130, 350], 'size': [320, 250], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 9}, {'name': 'vae', 'type': 'VAE', 'link': 10}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [11]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8]}, {'id': 8, 'type': 'SaveImage', 'pos': [1490, 350], 'size': [320, 250], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 11}], 'outputs': [], 'properties': {'Node name for S&R': 'SaveImage'}, 'widgets_values': ['ImageStudio/Restyle']}], 'links': [[1, 1, 1, 2, 0, 'CLIP'], [2, 1, 1, 3, 0, 'CLIP'], [3, 4, 0, 5, 0, 'IMAGE'], [4, 1, 2, 5, 1, 'VAE'], [5, 1, 0, 6, 0, 'MODEL'], [6, 2, 0, 6, 1, 'CONDITIONING'], [7, 3, 0, 6, 2, 'CONDITIONING'], [8, 5, 0, 6, 3, 'LATENT'], [9, 6, 0, 7, 0, 'LATENT'], [10, 1, 2, 7, 1, 'VAE'], [11, 7, 0, 8, 0, 'IMAGE']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.65, 'offset': [50, 100]}}, 'version': 0.4}, 'Inpaint': {'last_node_id': 8, 'last_link_id': 12, 'nodes': [{'id': 1, 'type': 'CheckpointLoaderSimple', 'pos': [0, 0], 'size': [320, 250], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [6]}, {'name': 'CLIP', 'type': 'CLIP', 'links': [1, 2]}, {'name': 'VAE', 'type': 'VAE', 'links': [5, 11]}], 'properties': {'Node name for S&R': 'CheckpointLoaderSimple'}, 'widgets_values': ['Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors']}, {'id': 2, 'type': 'CLIPTextEncode', 'pos': [380, 0], 'size': [320, 250], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [7]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': ['Cinematic photograph of a lone explorer overlooking a vast alien coastline, dramatic clouds, natural light, detailed textures, thoughtful composition'], 'title': 'Describe your image'}, {'id': 3, 'type': 'CLIPTextEncode', 'pos': [380, 300], 'size': [320, 250], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [8]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': ['blurry, low quality, distorted anatomy, watermark, oversaturated'], 'title': 'Negative prompt'}, {'id': 4, 'type': 'LoadImage', 'pos': [0, 400], 'size': [320, 250], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [4]}, {'name': 'MASK', 'type': 'MASK', 'links': [3]}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'Upload image; right-click to open MaskEditor'}, {'id': 5, 'type': 'VAEEncodeForInpaint', 'pos': [380, 650], 'size': [320, 250], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 4}, {'name': 'vae', 'type': 'VAE', 'link': 5}, {'name': 'mask', 'type': 'MASK', 'link': 3}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [9]}], 'properties': {'Node name for S&R': 'VAEEncodeForInpaint'}, 'widgets_values': [6]}, {'id': 6, 'type': 'KSampler', 'pos': [770, 0], 'size': [320, 250], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 6}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 7}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 8}, {'name': 'latent_image', 'type': 'LATENT', 'link': 9}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [10]}], 'properties': {'Node name for S&R': 'KSampler'}, 'widgets_values': [42, 'randomize', 30, 5.5, 'dpmpp_2m', 'karras', 1.0]}, {'id': 7, 'type': 'VAEDecodeTiled', 'pos': [1130, 350], 'size': [320, 250], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 10}, {'name': 'vae', 'type': 'VAE', 'link': 11}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [12]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8]}, {'id': 8, 'type': 'SaveImage', 'pos': [1490, 350], 'size': [320, 250], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 12}], 'outputs': [], 'properties': {'Node name for S&R': 'SaveImage'}, 'widgets_values': ['ImageStudio/Inpaint']}], 'links': [[1, 1, 1, 2, 0, 'CLIP'], [2, 1, 1, 3, 0, 'CLIP'], [3, 4, 1, 5, 2, 'MASK'], [4, 4, 0, 5, 0, 'IMAGE'], [5, 1, 2, 5, 1, 'VAE'], [6, 1, 0, 6, 0, 'MODEL'], [7, 2, 0, 6, 1, 'CONDITIONING'], [8, 3, 0, 6, 2, 'CONDITIONING'], [9, 5, 0, 6, 3, 'LATENT'], [10, 6, 0, 7, 0, 'LATENT'], [11, 1, 2, 7, 1, 'VAE'], [12, 7, 0, 8, 0, 'IMAGE']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.65, 'offset': [50, 100]}}, 'version': 0.4}, 'Detail': {'last_node_id': 9, 'last_link_id': 14, 'nodes': [{'id': 1, 'type': 'CheckpointLoaderSimple', 'pos': [0, 0], 'size': [320, 250], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [3, 8]}, {'name': 'CLIP', 'type': 'CLIP', 'links': [1, 2]}, {'name': 'VAE', 'type': 'VAE', 'links': [13]}], 'properties': {'Node name for S&R': 'CheckpointLoaderSimple'}, 'widgets_values': ['Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors']}, {'id': 2, 'type': 'CLIPTextEncode', 'pos': [380, 0], 'size': [320, 250], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [4, 9]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': ['Cinematic photograph of a lone explorer overlooking a vast alien coastline, dramatic clouds, natural light, detailed textures, thoughtful composition'], 'title': 'Describe your image'}, {'id': 3, 'type': 'CLIPTextEncode', 'pos': [380, 300], 'size': [320, 250], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [5, 10]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': ['blurry, low quality, distorted anatomy, watermark, oversaturated'], 'title': 'Negative prompt'}, {'id': 4, 'type': 'EmptyLatentImage', 'pos': [380, 650], 'size': [320, 250], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [6]}], 'properties': {'Node name for S&R': 'EmptyLatentImage'}, 'widgets_values': [1024, 1024, 1]}, {'id': 5, 'type': 'KSampler', 'pos': [770, 0], 'size': [320, 250], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 3}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 4}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 5}, {'name': 'latent_image', 'type': 'LATENT', 'link': 6}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [7]}], 'properties': {'Node name for S&R': 'KSampler'}, 'widgets_values': [42, 'randomize', 30, 5.5, 'dpmpp_2m', 'karras', 1.0]}, {'id': 6, 'type': 'LatentUpscaleBy', 'pos': [1130, 0], 'size': [320, 250], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 7}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [11]}], 'properties': {'Node name for S&R': 'LatentUpscaleBy'}, 'widgets_values': ['bislerp', 1.5]}, {'id': 7, 'type': 'KSampler', 'pos': [1490, 0], 'size': [320, 250], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 8}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 9}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 10}, {'name': 'latent_image', 'type': 'LATENT', 'link': 11}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [12]}], 'properties': {'Node name for S&R': 'KSampler'}, 'widgets_values': [42, 'fixed', 20, 5.0, 'dpmpp_2m', 'karras', 0.3], 'title': 'Detail pass — 1536px; costs more VRAM'}, {'id': 8, 'type': 'VAEDecodeTiled', 'pos': [1900, 350], 'size': [320, 250], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 12}, {'name': 'vae', 'type': 'VAE', 'link': 13}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [14]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8]}, {'id': 9, 'type': 'SaveImage', 'pos': [2260, 350], 'size': [320, 250], 'flags': {}, 'order': 8, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 14}], 'outputs': [], 'properties': {'Node name for S&R': 'SaveImage'}, 'widgets_values': ['ImageStudio/Detail']}], 'links': [[1, 1, 1, 2, 0, 'CLIP'], [2, 1, 1, 3, 0, 'CLIP'], [3, 1, 0, 5, 0, 'MODEL'], [4, 2, 0, 5, 1, 'CONDITIONING'], [5, 3, 0, 5, 2, 'CONDITIONING'], [6, 4, 0, 5, 3, 'LATENT'], [7, 5, 0, 6, 0, 'LATENT'], [8, 1, 0, 7, 0, 'MODEL'], [9, 2, 0, 7, 1, 'CONDITIONING'], [10, 3, 0, 7, 2, 'CONDITIONING'], [11, 6, 0, 7, 3, 'LATENT'], [12, 7, 0, 8, 0, 'LATENT'], [13, 1, 2, 8, 1, 'VAE'], [14, 8, 0, 9, 0, 'IMAGE']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.65, 'offset': [50, 100]}}, 'version': 0.4}}
COMFY_REV='72212fef660bcd7d9702fa52011d089c027a64d8'
MODEL='Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors'
MODEL_SHA='c9e3e68f89b8e38689e1097d4be4573cf308de4e3fd044c64ca697bdb4aa8bca'
MODEL_SIZE=7105348188
CF_SHA='1f99fb518b83dec2ba8d870a7cc991d82a77e6196295a3d2754ba17ab9f91872'
BASE=Path('/kaggle/temp/image-studio-v1'); BASE.mkdir(parents=True,exist_ok=True)
OUT=Path('/kaggle/working/image-studio'); OUT.mkdir(parents=True,exist_ok=True)
lock=open(BASE/'startup.lock','w')
try: fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
except BlockingIOError: raise RuntimeError('Image Studio is already running. Stop its startup cell before starting again.')
children=[]
def run(args,**kw):subprocess.run(list(map(str,args)),check=True,**kw)
def sha(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for chunk in iter(lambda:f.read(8*1024*1024),b''):h.update(chunk)
 return h.hexdigest()
def download(url,path,expected,size=None):
 if path.exists() and sha(path)==expected:return
 if path.exists():raise RuntimeError(f'Checksum mismatch: {path}. Preserve or move this file before retrying.')
 if size and shutil.disk_usage(path.parent).free<size+2*1024**3:raise RuntimeError('Insufficient free disk for model plus 2 GiB reserve.')
 temp=path.with_suffix(path.suffix+'.partial')
 print('Downloading',path.name,flush=True)
 with urllib.request.urlopen(url,timeout=120) as response,open(temp,'wb') as target:shutil.copyfileobj(response,target,8*1024*1024)
 if sha(temp)!=expected:raise RuntimeError(f'Download checksum mismatch: {temp}')
 temp.replace(path)
def spawn(args,log):
 f=open(log,'w');p=subprocess.Popen(list(map(str,args)),stdout=f,stderr=subprocess.STDOUT,start_new_session=True);f.close();children.append(p);return p
try:
 import torch
 if not torch.cuda.is_available():raise RuntimeError('Select a Kaggle GPU accelerator and restart this cell.')
 print('GPU:',torch.cuda.get_device_name(0),'Torch:',torch.__version__,flush=True)
 free,total=torch.cuda.mem_get_info(0)
 if free<10*1024**3:raise RuntimeError('GPU 0 has less than 10 GiB free. Stop other GPU notebooks/processes first.')
 # Separate T4s do not form one shared VRAM pool. This studio uses GPU 0.
 comfy=BASE/'ComfyUI'
 if not comfy.exists():
  run(['git','clone','--no-checkout','https://github.com/comfyanonymous/ComfyUI.git',comfy])
  run(['git','-C',comfy,'checkout','--detach',COMFY_REV])
 if subprocess.check_output(['git','-C',str(comfy),'rev-parse','HEAD'],text=True).strip()!=COMFY_REV:raise RuntimeError('Unexpected ComfyUI revision.')
 venv=BASE/'venv';python=venv/'bin/python'
 if not python.exists():run([sys.executable,'-m','venv','--system-site-packages','--without-pip',venv])
 constraints=BASE/'constraints.txt'
 import importlib.metadata as meta
 protected=['torch','torchvision','torchaudio','numpy','scipy']
 constraints.write_text('\n'.join(f'{p}=={meta.version(p)}' for p in protected))
 requirements=BASE/'requirements.txt'
 requirements.write_text('\n'.join(x for x in (comfy/'requirements.txt').read_text().splitlines() if x.strip() not in protected)+'\ntransformers==4.57.6\nhuggingface-hub==0.36.2\n')
 fingerprint=hashlib.sha256((requirements.read_text()+constraints.read_text()+sys.version).encode()).hexdigest()
 marker=BASE/'setup.complete'
 if not marker.exists() or marker.read_text()!=fingerprint:
  run([sys.executable,'-m','pip','--python',python,'install','-r',requirements,'-c',constraints])
  run([python,'-c','import torch, torchvision, transformers, aiohttp, safetensors; assert torch.cuda.is_available()'])
  marker.write_text(fingerprint)
 run([sys.executable,'-m','pip','--python',python,'freeze'],stdout=open(OUT/'packages.txt','w'))
 checkpoints=comfy/'models/checkpoints';checkpoints.mkdir(parents=True,exist_ok=True)
 # Reuse a checksum-verified checkpoint from any attached Kaggle input.
 attached=list(Path('/kaggle/input').rglob(MODEL)) if Path('/kaggle/input').exists() else []
 model_path=checkpoints/MODEL
 if not model_path.exists():
  for candidate in attached:
   if candidate.stat().st_size==MODEL_SIZE and sha(candidate)==MODEL_SHA:
    model_path.symlink_to(candidate);print('Reusing attached model:',candidate);break
 if not model_path.exists():
  download('https://huggingface.co/RunDiffusion/Juggernaut-XL-v9/resolve/main/'+MODEL,model_path,MODEL_SHA,MODEL_SIZE)
 if sha(model_path)!=MODEL_SHA:raise RuntimeError('Model verification failed.')
 workflow_dir=comfy/'user/default/workflows';workflow_dir.mkdir(parents=True,exist_ok=True)
 for name,wf in WORKFLOW_PAYLOAD.items():(workflow_dir/f'Image Studio - {name}.json').write_text(json.dumps(wf,indent=2))
 # Stable output destination: PNGs retain the actual graph, prompt, seed and settings.
 (OUT/'provenance.json').write_text(json.dumps({'comfy_revision':COMFY_REV,'model':MODEL,'sha256':MODEL_SHA,'torch':torch.__version__},indent=2))
 cf=BASE/'cloudflared'
 download('https://github.com/cloudflare/cloudflared/releases/download/2026.10.7/cloudflared-linux-amd64',cf,CF_SHA)
 cf.chmod(0o700)
 server=spawn([python,comfy/'main.py','--listen','127.0.0.1','--port','8188','--lowvram','--preview-method','none','--output-directory',OUT/'images','--disable-auto-launch'],OUT/'comfy.log')
 deadline=time.monotonic()+180
 while True:
  if server.poll() is not None:raise RuntimeError('ComfyUI stopped. See comfy.log in Kaggle outputs.')
  try:
   with urllib.request.urlopen('http://127.0.0.1:8188/object_info',timeout=3) as r:info=json.load(r)
   break
  except OSError:
   if time.monotonic()>deadline:raise RuntimeError('ComfyUI startup timed out. See comfy.log.')
   time.sleep(1)
 # Validate installed node classes before offering the browser link.
 for wf in WORKFLOW_PAYLOAD.values():
  for node in wf['nodes']:
   if node['type'] not in info:raise RuntimeError('Required node unavailable: '+node['type'])
 tunnel=spawn([cf,'tunnel','--url','http://127.0.0.1:8188','--no-autoupdate'],OUT/'tunnel.log')
 deadline=time.monotonic()+90
 url=None
 while time.monotonic()<deadline:
  text=(OUT/'tunnel.log').read_text()
  matches=re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com',text)
  if matches:url=matches[-1];break
  if tunnel.poll() is not None:raise RuntimeError('Cloudflare stopped; inspect tunnel.log.')
  time.sleep(1)
 if not url:raise RuntimeError('No Cloudflare URL received; inspect tunnel.log.')
 print('\nOPEN IMAGE STUDIO:',url,flush=True)
 print('Workflow menu → Image Studio - Create / Restyle / Inpaint / Detail. Edit prompt, then press Run.')
 print('No generation has started. Keep this cell running. Anyone with this session URL can access the studio.')
 print('Save Version with outputs before ending Kaggle. Images:',OUT/'images',flush=True)
 while server.poll() is None and tunnel.poll() is None:time.sleep(2)
 raise RuntimeError('Studio or tunnel stopped; see the saved logs.')
finally:
 for p in reversed(children):
  if p.poll() is None:
   os.killpg(p.pid,signal.SIGTERM)
   try:p.wait(timeout=10)
   except subprocess.TimeoutExpired:os.killpg(p.pid,signal.SIGKILL);p.wait()
 fcntl.flock(lock,fcntl.LOCK_UN);lock.close()
